In [6]:
LABELS = ['Colleague', 'Engaged', 'Excited', 'EyeContact', 'Smiled', 'Calm']
participant_ids =['P1', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P20', 'P21', 'P22', 'P24', 'P25', 'P27', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P37', 'P42', 'P43', 'P44', 'P45', 'P47', 'P48', 'P49', 'P50', 'P52', 'P53', 'P55', 'P56', 'P57', 'P58', 'P59', 'P60', 'P61', 'P62', 'P63', 'P64', 'P65', 'P66', 'P67', 'P69', 'P70', 'P71', 'P72', 'P73', 'P74', 'P76', 'P77', 'P78', 'P79', 'P80', 'P81', 'P83', 'P84', 'P85', 'P86', 'P89', 'PP1', 'PP3', 'PP4', 'PP5', 'PP6', 'PP7', 'PP8', 'PP10', 'PP11', 'PP12', 'PP13', 'PP14', 'PP15', 'PP16', 'PP17', 'PP20', 'PP21', 'PP22', 'PP24', 'PP25', 'PP27', 'PP29', 'PP30', 'PP31', 'PP32', 'PP33', 'PP34', 'PP35', 'PP37', 'PP42', 'PP43', 'PP44', 'PP45', 'PP47', 'PP48', 'PP49', 'PP50', 'PP52', 'PP53', 'PP55', 'PP56', 'PP57', 'PP58', 'PP59', 'PP60', 'PP61', 'PP62', 'PP63', 'PP64', 'PP65', 'PP66', 'PP67', 'PP69', 'PP70', 'PP71', 'PP72', 'PP73', 'PP74', 'PP76', 'PP77', 'PP78', 'PP79', 'PP80', 'PP81', 'PP83', 'PP84', 'PP85', 'PP86', 'PP89']
FRAMES_BATCH_SIZE = 8

In [7]:
import gc
import cv2
import numpy as np
from tensorflow.keras.utils import to_categorical
from hireverse.utils.dataset_handler import DatasetHandler


def participant_frames_generator(participant_id, number_of_frames_in_batch=16):
    total_frames = DatasetHandler.get_number_of_frames(participant_id)
    frame_generator = DatasetHandler.yield_sorted_participant_frames_images(participant_id, is_image_greyscale=True)
    label_dict = DatasetHandler.get_labels_dict(participant_id)

    for _ in range(0, total_frames, number_of_frames_in_batch):
        frames_batch = []
        for __ in range(number_of_frames_in_batch):
            try:
                frame = next(frame_generator)
                frame = frame.astype('float32') / 255.0  # Normalize
                frames_batch.append(frame)
            except StopIteration:
                break

        if frames_batch:  # Yield only if we have at least 1 frame
            yield frames_batch, {comp: to_100class(label_dict[comp]) for comp in LABELS}


def to_100class(score_1_to_10):
    return np.floor((score_1_to_10 - 1) * 10 + np.random.uniform(0, 10))

def train_generator(train_ids, number_of_videos_in_the_batch=6):
    participants_frame_gens = {participant_id: participant_frames_generator(participant_id, number_of_frames_in_batch=FRAMES_BATCH_SIZE) for participant_id in train_ids}

    while True:  # Loop indefinitely for continuous training
        selected_ids = np.random.choice(train_ids, size=number_of_videos_in_the_batch, replace=False)
        X_batch = []
        y_batch = {comp: [] for comp in LABELS}
        selected_ids = np.random.choice(train_ids, 6, replace=False)

        for participant_id in selected_ids:
            participant_frame_gen = participants_frame_gens[participant_id]
            try:
                frames, labels = next(participant_frame_gen)
                X_batch.extend(frames)
                for comp in LABELS:
                    y_batch[comp].extend([labels[comp]] * len(frames))

            except StopIteration:
                participants_frame_gens[participant_id] = participant_frames_generator(participant_id, number_of_frames_in_batch=FRAMES_BATCH_SIZE)    # TODO: why?
                continue

        X_batch = np.array(X_batch)
        idx = np.random.permutation(len(X_batch))
        yield X_batch[idx], {comp: np.array(y_batch[comp])[idx] for comp in LABELS}


In [8]:
from sklearn.model_selection import train_test_split

# TODO: use group split
train_ids, temp_ids = train_test_split(participant_ids, test_size=0.5, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

train_gen = train_generator(train_ids)

In [9]:
X, y = next(train_gen)
print(X.shape)  # Should be: (96, 640, 640, 1)
# [
#     array([[...]],  # Frame 1 for participant 1
#     array([[...]],  # Frame 2 for participant 1
#     ...
#     array([[...]],  # Frame 1 for participant 2
#     array([[...]],  # Frame 2 for participant 2
#     ...
# ]

# print(y.shape)  # Should be: (6,) - but actually a dictionary of 6 arrays
# {
#     'hireability': [35, 67, 50, ..., 40],  # 96 scores for frames from participant 1, 2, etc.
#     'friendliness': [12, 45, 38, ..., 55],
#     'eye_contact': [87, 34, 55, ..., 91]
# }

(48, 640, 640)


In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_competency_cnn():
    # CHANGED: Paper's exact architecture (Section 3.5)
    inputs = tf.keras.Input(shape=(640, 640, 1))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(128, (3,3), activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(4096, activation='relu')(x)  # Paper's large dense layer
    x = layers.Dropout(0.5)(x)  # Paper's dropout rate

    # CHANGED: Multiple output heads (6 competencies)
    outputs = [layers.Dense(100, activation='softmax', name=comp)(x) for comp in LABELS]

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # CHANGED: Paper's training parameters (Section 3.5)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),  # Paper's LR
        loss={comp: 'sparse_categorical_crossentropy' for comp in LABELS},
        metrics=['accuracy'] * len(LABELS)
    )
    return model

# Initialize model
model = build_competency_cnn()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 640, 640,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 638, 638,  │        320 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 319, 319,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 317, 317,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 158, 158,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 156, 156,  │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 4096)      │    528,384 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 4096)      │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Colleague (Dense)   │ (None, 100)       │    409,700 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Engaged (Dense)     │ (None, 100)       │    409,700 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Excited (Dense)     │ (None, 100)       │    409,700 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ EyeContact (Dense)  │ (None, 100)       │    409,700 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Smiled (Dense)      │ (None, 100)       │    409,700 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Calm (Dense)        │ (None, 100)       │    409,700 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,079,256 (11.75 MB)

 Trainable params: 3,079,256 (11.75 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
verbose=0

In [12]:
model.fit(
    train_generator(train_ids),
    steps_per_epoch=len(train_ids) // 6,
    epochs=256,
    # validation_data=val_generator(val_ids),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=10),
        tf.keras.callbacks.ModelCheckpoint('best_model.h5')
    ]
)

Epoch 1/256


KeyboardInterrupt: 

Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Discarded (victim of GPU error/recovery) (00000005:kIOGPUCommandBufferCallbackErrorInnocentVictim)
	<AGXG13GFamilyCommandBuffer: 0x3da193ce0>
    label = <none> 
    device = <AGXG13GDevice: 0x12f15f400>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x14e6a0a00>
        label = <none> 
        device = <AGXG13GDevice: 0x12f15f400>
            name = Apple M1 
    retainedReferences = 1
